# SPK-4 — Disk Spill (too few shuffle partitions)

**Break → Detect → Fix → Prove.** A heavy shuffle (a wide aggregation / sort) split into **too
few** partitions makes each post-shuffle partition larger than the memory one task has to sort it
in — so the engine **spills to disk**. The job still finishes, but it crawls.

**Pre-requisite:** the unified Spark server is up (`make up`). This notebook connects via Spark
Connect. **Open the Spark UI at http://localhost:4040** and watch it while the cells run.

**Laptop-safe:** ~20M rows are *generated lazily* and only aggregated / `count()`-ed — never
collected or written — so nothing fills memory or disk beyond Spark's short-lived spill scratch.
This is a **partition-sizing** problem (per-task sort memory vs partition *size*), not a
container-memory one, so the default **tuned** box is fine — no need for `make up-constrained`.

See the companion writeup in [`README.md`](./README.md), the
[Spark-UI guide](../../docs/spark-ui-guide.md), and the [troubleshooting sheet](../../docs/troubleshooting.md).

In [ ]:
from common.spark_session import spark, display_df
from common.profiles import apply_profile, profile_summary
from common.datagen import uniform_keys, wide_rows
from common.metrics_diff import measure, compare
from pyspark.sql import functions as F

# Watch this while the cells run — spill shows up in Stages -> Tasks -> Summary Metrics
# (the "Spill (memory)" and "Spill (disk)" rows).
print("Spark UI:", "http://localhost:4040")
spark

## Step 0 — Parameters & the dataset to aggregate

We synthesize ~20M **wide-ish** rows lazily (more bytes per row → a heavier shuffle payload), with
a join/group key spread over many distinct values. Nothing is stored; cost is paid only on an
action. We'll roll this up with a `groupBy(many keys).agg(...)` + `orderBy` — a wide shuffle that
must sort each post-shuffle partition in memory.

In [ ]:
# Lower N_ROWS to 5-10M if your laptop is slow; raise it to make the spill more dramatic.
N_ROWS  = 20_000_000
N_KEYS  = 50_000       # many group-by keys -> a genuinely large shuffle to partition
N_COLS  = 12           # wide-ish rows -> more bytes per row -> heavier shuffle payload

# A wide fact: row_id + c0..c{N_COLS-1} doubles, plus a grouping key spread over N_KEYS values.
wide = wide_rows(spark, n_rows=N_ROWS, n_cols=N_COLS)
fact = wide.withColumn("group_key", F.pmod(F.col("row_id"), F.lit(N_KEYS)))

# The heavy aggregation we'll run under every profile: group by many keys, sum the wide columns,
# then sort. The groupBy forces a shuffle; sorting each post-shuffle partition is what spills when
# the partitions are too big. Sums across all N_COLS keep each task doing real sort/aggregate work.
agg_exprs = [F.sum(f"c{i}").alias(f"sum_c{i}") for i in range(N_COLS)]

def heavy_agg():
    return (fact.groupBy("group_key")
                .agg(*agg_exprs)
                .orderBy(F.desc("sum_c0")))

print(f"fact: ~{N_ROWS:,} wide rows ({N_COLS} value cols), {N_KEYS:,} group keys (all lazy)")

In [ ]:
# Sanity-check the shape (small result, safe to show): how many distinct keys we'll shuffle into.
print("distinct group keys:", fact.select("group_key").distinct().count())
fact.select("row_id", "group_key", "c0", "c1").show(5)

## 1. Break it — heavy aggregation into too FEW shuffle partitions

The `constrained` session profile pins **`spark.sql.shuffle.partitions = 16`** and turns **AQE
off**. The entire shuffle output is forced into 16 oversized partitions, so each sort/aggregate
task is handed far more data than it can sort in memory — Spark's external sort **spills to disk**.
AQE is off so `coalescePartitions` can't quietly resize them and hide the problem.

In [ ]:
apply_profile(spark, "constrained")   # AQE off + shuffle.partitions = 16
profile_summary(spark)

m_spilling = measure(spark, "16 partitions (spilling)", lambda: heavy_agg().count())
print("\nspilling-aggregation metrics:", m_spilling)

## 2. Detect it — read the Spark UI

http://localhost:4040 -> **SQL / DataFrame** -> click this query -> the aggregation's reduce
**Stage** -> **Tasks -> Summary Metrics**. The tell (see [`docs/spark-ui-guide.md`](../../docs/spark-ui-guide.md)):
non-zero **Spill (memory)** and especially **Spill (disk)**, on a reduce stage with only **16
tasks** each reading a large **Shuffle Read Size**. The cell below prints the same spill figures
`metrics_diff` captured — `spill_mem_bytes` / `spill_disk_bytes`.

In [ ]:
print(f"reduce-stage tasks   : {m_spilling['num_tasks']}   <- = shuffle.partitions (16): too few")
print(f"shuffle read         : {m_spilling['shuffle_read_bytes']} bytes  (split across only 16 partitions)")
print(f"SPILL (memory)       : {m_spilling['spill_mem_bytes']} bytes")
print(f"SPILL (disk)         : {m_spilling['spill_disk_bytes']} bytes   <- the expensive one: partitions too big to sort in memory")

## 3. Fix it (A) — raise `spark.sql.shuffle.partitions`

The direct lever: more post-shuffle partitions -> each is smaller -> it fits in a task's sort
memory -> no spill. We bump it to **200** while keeping **AQE off**, so this isolates the
partition-count effect (AQE isn't doing anything for us yet).

In [ ]:
apply_profile(spark, "constrained", **{"spark.sql.shuffle.partitions": "200"})
profile_summary(spark)

m_200 = measure(spark, "200 partitions", lambda: heavy_agg().count())
print("\n200-partition metrics:", m_200)

## 3. Fix it (B) — let AQE coalesce the partitions

The production default: `apply_profile(spark, "tuned")` turns **AQE on** (with
`coalescePartitions`). AQE reads the *actual* shuffle size at runtime and picks a partition count
from `advisoryPartitionSizeInBytes` — no hand-tuned magic number, and it adapts as data volume
changes.

In [ ]:
apply_profile(spark, "tuned")   # AQE on, coalescePartitions on -> partitions sized to the data
profile_summary(spark)

m_tuned = measure(spark, "AQE (tuned)", lambda: heavy_agg().count())
print("\nAQE-tuned metrics:", m_tuned)

## 4. Prove it

Before/after. Watch the **Spill (disk)** row collapse toward **0** as the partitions shrink to fit
in memory, with **Spill (memory)** and runtime falling alongside it.

In [ ]:
compare([m_spilling, m_200, m_tuned])

## Takeaways & "in real production..."

- **Detect** spill by the per-task **Spill (memory)** / **Spill (disk)** columns in Stages -> Tasks —
  disk spill is the costly one; non-zero is a red flag even when the job "succeeds".
- **Right-size the shuffle:** post-shuffle partition size ~= total shuffle bytes / `shuffle.partitions`.
  Aim for partitions that fit in per-task memory (rule of thumb ~100-200 MB).
- **Prefer AQE** (`coalescePartitions` + `advisoryPartitionSizeInBytes`) so the partition count
  tracks real data size instead of a hard-coded number that's wrong half the time.
- **The tradeoff (don't over-correct):** raising `shuffle.partitions` too far swings into the
  *opposite* pathology — thousands of tiny partitions where **scheduling overhead** dominates a
  small job. That's **`SPK-9`** (shuffle internals & stages). There's a sweet spot; AQE finds it.
- **In prod:** alert on non-zero disk spill on hot stages, keep AQE enabled, and set a sensible
  `advisoryPartitionSizeInBytes` rather than pinning `shuffle.partitions` globally.

## Teardown

Nothing durable was written (we only aggregated and counted generated data), so there is nothing
to delete. We restore the production-tuned safety nets and clear any cache. Spark's transient
spill scratch lives under `.tmp/`; `make clean` clears it.

In [ ]:
apply_profile(spark, "tuned")        # restore production-tuned safety nets
spark.catalog.clearCache()
print("Done. Profile reset to 'tuned'. No tables/files were created; `make clean` clears .tmp if needed.")